# Tutorial 20: YOLO Multi-Channel C++ Demo

This tutorial explains how a C++ application runs YOLO object detection on multiple video channels with a DEEPX NPU. It also shows how to download the resources, build the application, and run the video or camera demo.

![YOLO multi-channel demo](assets/yolo-multi-sc.png)

## Learning goals

By the end of this tutorial, you will be able to:

- understand the C++ project structure;
- read the model, input, and display settings in the JSON files;
- follow the asynchronous multi-channel inference flow;
- build the application in Release mode; and
- run the video and camera examples.

## 1. Locate the tutorial files

The following cell finds the tutorial directory whether JupyterLab was started from the repository root or from this notebook directory.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "T20-demo-yolo-multi",
]

TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "app" / "CMakeLists.txt").is_file()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not find the T20-demo-yolo-multi directory.")

APP_ROOT = TUTORIAL_ROOT / "app"
ASSET_ROOT = TUTORIAL_ROOT / "assets"

print(f"Tutorial: {TUTORIAL_ROOT}")
print(f"App:      {APP_ROOT}")
print(f"Assets:   {ASSET_ROOT}")

### Project layout

```text
T20-demo-yolo-multi/
├── get_resources.sh
├── assets/
│   ├── models/
│   └── videos/
├── app/
│   ├── build.sh
│   ├── run_camera.sh
│   ├── run_video.sh
│   ├── config/
│   ├── include/
│   ├── src/
│   ├── lib/
│   ├── extern/
│   └── sample/
└── yolo_multi.ipynb
```

`extern/` contains the header-only cxxopts and RapidJSON dependencies. `sample/` contains fonts and images used by the display UI.

In [ ]:
required_files = [
    TUTORIAL_ROOT / "get_resources.sh",
    APP_ROOT / "build.sh",
    APP_ROOT / "run_camera.sh",
    APP_ROOT / "run_video.sh",
    APP_ROOT / "CMakeLists.txt",
    APP_ROOT / "config" / "ppu_yolo_multi_33_camera.json",
    APP_ROOT / "config" / "ppu_yolo_multi_36_video.json",
    APP_ROOT / "src" / "yolo_multi_channels.cpp",
    APP_ROOT / "src" / "od.cpp",
]

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{status:7}] {path.relative_to(TUTORIAL_ROOT)}")

## 2. Check the environment

Install the Debian packages listed in `README.md` before continuing. The DEEPX device driver and DXRT SDK must also be installed. This check does not install or change anything.

In [ ]:
commands = [
    "g++",
    "cmake",
    "make",
    "curl",
    "tar",
    "gst-inspect-1.0",
    "dxrt-cli",
    "dxparse",
    "dxrun",
]

for command in commands:
    location = shutil.which(command)
    print(f"[{'OK' if location else 'MISSING':7}] {command}")

device_nodes = sorted(Path("/dev").glob("dxrt*"))
print(f"\nDEEPX device nodes: {device_nodes if device_nodes else 'not found'}")

## 3. Download and check the resources

Both demos use `assets/models/YOLOV5S_PPU.dxnn` and sample MP4 files under `assets/videos/`. `get_resources.sh` downloads one archive, extracts it into `assets/`, and removes the archive after successful extraction.

In [ ]:
config_paths = [
    APP_ROOT / "config" / "ppu_yolo_multi_33_camera.json",
    APP_ROOT / "config" / "ppu_yolo_multi_36_video.json",
]

required_resources = set()
for config_path in config_paths:
    data = json.loads(config_path.read_text(encoding="utf-8"))
    required_resources.add((APP_ROOT / data["model_path"]).resolve())
    for source in data["video_sources"]:
        if not source[0].startswith("/dev/"):
            required_resources.add((APP_ROOT / source[0]).resolve())

missing_resources = sorted(path for path in required_resources if not path.is_file())
print(f"Required resource files: {len(required_resources)}")
print(f"Missing resource files:  {len(missing_resources)}")
for path in missing_resources[:10]:
    print(f"  - {path}")

Run the next cell only when resources are missing. Existing files with the same names may be replaced during extraction.

In [ ]:
if missing_resources:
    subprocess.run(
        [str(TUTORIAL_ROOT / "get_resources.sh")],
        cwd=TUTORIAL_ROOT,
        check=True,
    )
else:
    print("All required resources are already available.")

### 3.1. Inspect the model with `dxparse`

Use `dxparse` to inspect the model structure, task graph, tensors, memory use, and dependencies:

```bash
dxparse -m assets/models/YOLOV5S_PPU.dxnn -v
```

The verbose output reports the following model input:

```text
images {shape: [1, 512, 512, 3], dtype: uint8}
```

The dimensions are batch, height, width, and channels. Therefore, this model expects a 512 x 512 three-channel input, not a 640 x 640 input.

In [ ]:
MODEL_PATH = ASSET_ROOT / "models" / "YOLOV5S_PPU.dxnn"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
if shutil.which("dxparse") is None:
    raise FileNotFoundError("dxparse is not installed or is not in PATH.")

subprocess.run(
    ["dxparse", "-m", "assets/models/YOLOV5S_PPU.dxnn", "-v"],
    cwd=TUTORIAL_ROOT,
    check=True,
)

### 3.2. Benchmark the model with `dxrun`

Run a five-second CLI benchmark with automatically generated dummy input:

```bash
dxrun -m assets/models/YOLOV5S_PPU.dxnn --use-ort -t 5
```

`dxrun` uses benchmark mode by default when neither `--single` nor `--fps` is specified. `-t 5` sets the measurement duration to five seconds, and `--use-ort` enables ONNX Runtime for CPU tasks in the model graph. The reported result provides a command-line performance baseline without multi-channel video decoding or display overhead.

In [ ]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
if shutil.which("dxrun") is None:
    raise FileNotFoundError("dxrun is not installed or is not in PATH.")

subprocess.run(
    [
        "dxrun",
        "-m",
        "assets/models/YOLOV5S_PPU.dxnn",
        "--use-ort",
        "-t",
        "5",
    ],
    cwd=TUTORIAL_ROOT,
    check=True,
)

## 4. Understand the demo configurations

The application reads all runtime settings from a JSON file. Important fields are:

- `model_path`: path to the DXNN model, relative to `app/`;
- `model_name`: selects the matching YOLO post-processing parameters;
- `video_sources`: input path, input type, and optional saved-frame count;
- `display_config`: output size, grid, FPS, and layout settings; and
- `num_devices`: number of NPU devices shown in the header.

In [ ]:
for config_path in config_paths:
    data = json.loads(config_path.read_text(encoding="utf-8"))
    display = data["display_config"]
    camera_count = sum(source[1] == "camera" for source in data["video_sources"])
    print(config_path.name)
    print(f"  model:      {data['model_name']}")
    print(f"  channels:   {len(data['video_sources'])}")
    print(f"  cameras:    {camera_count}")
    print(f"  output:     {display['output_width']} x {display['output_height']}")
    print(f"  grid:       {display['grid_cols']} x {display['grid_rows']}")
    print(f"  expand:     {display['expand_mode']}")
    print()

The camera configuration contains one `/dev/video0` input and 32 video inputs. Camera highlighting is independent of `expand_mode`, so the camera is placed in an enlarged center area with a yellow border.

The video configuration contains 36 video inputs in a 6 x 6 grid. Although its JSON sets `expand_mode` to `true`, the current C++ code enables that special layout only for 33, 41, 61, or 73 sources. It therefore uses the standard 6 x 6 layout for 36 sources.

## 5. Read the C++ code

The notebook displays selected sections directly from the current source files. This avoids keeping a second copy of the C++ code in the notebook.

In [ ]:
from IPython.display import Code, display

def show_source(relative_path, marker, line_count=80):
    path = APP_ROOT / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    try:
        start = next(index for index, line in enumerate(lines) if marker in line)
    except StopIteration as error:
        raise ValueError(f"Marker not found in {relative_path}: {marker}") from error

    end = min(start + line_count, len(lines))
    print(f"{relative_path}:{start + 1}-{end}")
    display(Code("\n".join(lines[start:end]), language="cpp"))

### 5.1. Build configuration

CMake builds one C++14 executable and links DXRT, OpenCV, pthread, OpenMP, and the filesystem library required by C++14. OpenCV FreeType support is used when available.

In [ ]:
show_source("CMakeLists.txt", "add_dxrt_lib()", line_count=85)

### 5.2. Configuration parsing

`ApplicationJsonParser()` validates the JSON fields and fills `AppConfig`. The parser also supplies defaults for optional display settings.

In [ ]:
show_source("src/yolo_multi_channels.cpp", "struct AppConfig", line_count=32)
show_source("src/yolo_multi_channels.cpp", "int ApplicationJsonParser", line_count=70)

### 5.3. Multi-channel inference pipeline

```text
JSON configuration
        |
        v
DXRT InferenceEngine (shared)
        |
        +-- ObjectDetection: channel 1 --+
        +-- ObjectDetection: channel 2 --+--> output grid --> OpenCV window
        +-- ObjectDetection: channel N --+
```

`main()` creates one DXRT `InferenceEngine` and one `ObjectDetection` object per input source. Each channel runs in its own worker thread.

In [ ]:
show_source(
    "src/yolo_multi_channels.cpp",
    "auto ie = std::make_shared<dxrt::InferenceEngine>",
    line_count=45,
)

### 5.4. Per-channel asynchronous inference

`ObjectDetection::threadFunc()` gets a preprocessed input frame and calls `RunAsync()`. The callback updates the latest bounding boxes while the channel thread prepares the display frame. Mutexes protect data shared between the worker and callback.

In [ ]:
show_source("src/od.cpp", "void ObjectDetection::threadFunc", line_count=48)
show_source("src/od.cpp", "void ObjectDetection::PostProc", line_count=18)

### 5.5. YOLO post-processing

`Yolo::PostProc()` selects the decoder for the model output type. These demos use the PPU output path. The decoded candidates are sorted by class and passed to non-maximum suppression to remove overlapping boxes.

In [ ]:
show_source("src/demo_utils/yolo.cpp", "std::vector<BoundingBox> Yolo::PostProc", line_count=58)

## 6. Build the application

`build.sh` creates `app/build/`, configures CMake in Release mode, and runs `make` with all CPU cores reported by `nproc`. Use `./build.sh --clean` when a full rebuild is required.

In [ ]:
build_result = subprocess.run(
    ["./build.sh"],
    cwd=APP_ROOT,
    check=False,
)

binary_path = APP_ROOT / "build" / "yolo_multi_demo"
print(f"Build exit code: {build_result.returncode}")
print(f"Executable exists: {binary_path.is_file()}")

## 7. Run the 36-channel video demo

The next cell opens an OpenCV window and waits until the demo exits. Press `Esc` or `q`, or click the `EXIT` button.

In [ ]:
if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")

subprocess.run(["./run_video.sh"], cwd=APP_ROOT, check=False)

## 8. Run the 33-channel camera demo

The default configuration expects a V4L2 camera at `/dev/video0`. The application selects a supported camera mode automatically and limits the camera rate to at most 30 FPS.

In [ ]:
camera_path = Path("/dev/video0")
print(f"Camera exists: {camera_path.exists()}")
if shutil.which("v4l2-ctl"):
    subprocess.run(["v4l2-ctl", "--list-devices"], check=False)

In [ ]:
if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if not camera_path.exists():
    raise FileNotFoundError("The camera configuration requires /dev/video0.")

subprocess.run(["./run_camera.sh"], cwd=APP_ROOT, check=False)

## Controls

- `Esc` or `q`: exit
- `t`: show or hide detection boxes
- `EXIT` button: exit with the mouse

## Troubleshooting

- **DXRT is not found:** verify the DXRT SDK installation and run `dxrt-cli -s`.
- **A model or video cannot be opened:** check the paths under `assets/` and run `get_resources.sh` again.
- **The camera cannot be opened:** verify `/dev/video0` and the current user's `video` group permission.
- **The window does not appear:** use a graphical desktop, remote desktop, or correctly configured X11 forwarding.
- **The notebook cell remains busy:** the run cell blocks while the OpenCV event loop is active; exit the demo window to finish the cell.

## Summary

The application shares one DXRT inference engine across multiple channel workers. Each worker performs asynchronous inference, YOLO PPU post-processing, and frame rendering. The main loop combines the channel frames into one configurable grid and displays runtime information in an OpenCV window.